In [ ]:
import sqlite3
import pandas as pd
#for nicer table display
from IPython.display import display


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
conn = sqlite3.connect("hospital_system.db")  # database file
conn.execute("PRAGMA foreign_keys = ON;")     # enable foreign keys
cursor = conn.cursor()


In [ ]:
# Drop tables if they exist
cursor.execute("DROP TABLE IF EXISTS billing;")
cursor.execute("DROP TABLE IF EXISTS visits;")
cursor.execute("DROP TABLE IF EXISTS patients;")
conn.commit()

In [ ]:
# Create Patients table
cursor.execute("""
CREATE TABLE patients (
    patient_id INTEGER PRIMARY KEY,
    age INTEGER CHECK(age >= 0 AND age <= 120),
    gender TEXT,
    city TEXT,
    insurance_provider TEXT,
    chronic_flag BOOLEAN,
    registration_date DATE
)
""")

# Create Visits table
cursor.execute("""
CREATE TABLE visits (
    visit_id INTEGER PRIMARY KEY,
    patient_id INTEGER NOT NULL,
    visit_date DATE NOT NULL,
    department TEXT,
    visit_type TEXT,
    length_of_stay_hours REAL,
    risk_score TEXT,
    doctor_id INTEGER,
    FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
)
""")

# Create Billing table
cursor.execute("""
CREATE TABLE billing (
    bill_id INTEGER PRIMARY KEY,
    visit_id INTEGER NOT NULL,
    billed_amount REAL,
    approved_amount REAL,
    claim_status TEXT,
    payment_days INTEGER,
    billing_date DATE,
    FOREIGN KEY(visit_id) REFERENCES visits(visit_id)
)
""")

conn.commit()
print("Tables created successfully!")


In [ ]:
# Loading CSV into pandas
patients_df = pd.read_csv("patients.csv")
visits_df = pd.read_csv("visits.csv")
billing_df = pd.read_csv("billing.csv")

# Loading into SQLite
patients_df.to_sql("patients", conn, if_exists="append", index=False)
visits_df.to_sql("visits", conn, if_exists="append", index=False)
billing_df.to_sql("billing", conn, if_exists="append", index=False)


In [ ]:
cursor.execute("CREATE INDEX idx_visit_date ON visits(visit_date)")
cursor.execute("CREATE INDEX idx_department ON visits(department)")
cursor.execute("CREATE INDEX idx_insurance_provider ON patients(insurance_provider)")
cursor.execute("CREATE INDEX idx_claim_status ON billing(claim_status)")
cursor.execute("CREATE INDEX idx_patient_id ON visits(patient_id)")
cursor.execute("CREATE INDEX idx_visit_id ON billing(visit_id)")
conn.commit()


1. Index on visit_date is useful as it:
- Speeds up time-based queries
- Important for monthly/quarterly trend analysis
- Supports time-based validation in ML

- Hospital leadership often monitors patient volume and revenue trends over time. Indexing visit_date ensures fast retrieval of date-filtered records.

2. Index on department is useful as it:
- Imporves performance of department level
- Speeds up grouping and filtering queries

- Department performance is amkey operational KPI. Since many queries group by department, indexing reduces query execution time.

3. Index on insurance_provider is useful as it:
- Speeds up joins between pateitns and billing
- Improves provider-level revenue analysis.

- Insurance behavior directly impacts revenue leakage. frequent grouping and filtering by provider makes indexing essential.

4. Index on claim_status is useful as it:
- Speeds up rejection-rate queries.
- Improves performance when filtering by "rejected" or "pending"

- Finance teams constantly monitor rejected claims. Quick access to these records improves reporting efficiency.

5. Index on patient_id and visit_id is useful as it:
- Optimises JOIN operations.
- Reduces lookup time across tables.

- As I understand, entire relational model depends on joins. Without indexing these keys, queries would slow down significantly as data scales.


- The hospital analytics layer was implemente dusing SQLite within the Google Colab environment. SQLite is a fully relational, SQL-compliant database engine that supports primary keys, foreign keys, indexing keys, joins, and constraints. It enables structured relational modeling while maintaing lightweight deployment suitable for academic and anlytical workflows.

- Foreign key enforcement was explicitly enabled to maintain referential integrity between patients, visits, and billing tables. This ensures that visits cannot exist without a valid patient and billing records cannot exist withut a corresponding visit. Enforcing referential integrity improves data reliability and orevents orphan records, which is critical in healthcare systems.

- Indexes were created on high-frequency query columns including visit_date, department, insurance_provider, and claim_status.
These fields are commonly used in filtering, grouping, and aggregation queries within operational and financial reporting. The indexing strategy improves query efficiency and simulates production-grade database optimization practices.

- The SQL analytics layer serves as the structured data foundation for downstream analytics and machine learning workflows. All feature engineering and modeling datasets are extracted using SQL queries, ensuring consistency, integrity, and traceability

In [ ]:
# OPERATIONAL ANALYSIS
# Top 10 Departments by Total Visit Volume

pd.read_sql_query("""
SELECT department,
       COUNT(*) AS total_visits
FROM visits
GROUP BY department
ORDER BY total_visits DESC
LIMIT 10;
""", conn)


Departments with high visit volumes may require additional staffing or resource allocation.

In [ ]:
# Top 5 Departments with Highest Average Length of Stay

pd.read_sql_query("""
SELECT department,
       ROUND(AVG(length_of_stay_hours), 2) AS avg_length_of_stay
FROM visits
GROUP BY department
ORDER BY avg_length_of_stay DESC
LIMIT 5;
""", conn)

In [ ]:
# Percentage of High-Risk Visits per Department

pd.read_sql_query("""
SELECT department,
       ROUND(100.0 *
             SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END)
             / COUNT(*), 2) AS high_risk_percentage
FROM visits
GROUP BY department
ORDER BY high_risk_percentage DESC;
""", conn)

Departments with high-risk concentrations may need enhanced monitoring.

In [ ]:
# Average number of visists per patient by City

pd.read_sql_query("""
SELECT p.city,
       ROUND(COUNT(v.visit_id) * 1.0
             / COUNT(DISTINCT p.patient_id), 2) AS avg_visits_per_patient
FROM patients p
JOIN visits v ON p.patient_id = v.patient_id
GROUP BY p.city
ORDER BY avg_visits_per_patient DESC;
""", conn)

Higher repeat visit rates may signal chronic disease.

In [ ]:
# Doctors handling the most high-risk visits
pd.read_sql_query("""
SELECT doctor_id,
COUNT(*) AS high_risk_visits
FROM visits
WHERE risk_score = 'High'
GROUP BY doctor_id
ORDER BY high_risk_visits DESC;
""", conn)

In [ ]:
# FINANCIAL ANALYSIS
# Top 10 Insurance Providers by Total Billed Amount

pd.read_sql_query("""
SELECT p.insurance_provider,
       ROUND(SUM(b.billed_amount), 2) AS total_billed
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
JOIN patients p ON v.patient_id = p.patient_id
GROUP BY p.insurance_provider
ORDER BY total_billed DESC
LIMIT 10;
""", conn)

In [ ]:
# Top 5 Insurance Providers with Highest Claim Rejection Rate

pd.read_sql_query("""
SELECT p.insurance_provider,
       ROUND(100.0 *
             SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END)
             / COUNT(*), 2) AS rejection_rate
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
JOIN patients p ON v.patient_id = p.patient_id
GROUP BY p.insurance_provider
ORDER BY rejection_rate DESC
LIMIT 5;
""", conn)

In [ ]:
# Average PAyment Delay by Insuerance Provider
pd.read_sql_query("""
SELECT p.insurance_provider,
ROUND(AVG(b.payment_days), 2) AS avg_payment_delay
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
JOIN patients p ON v.patient_id = p.patient_id
GROUP BY p.insurance_provider
ORDER BY avg_payment_delay DESC;
""", conn)

In [ ]:
# Revenue Realization Ratio by Department
pd.read_sql_query("""
SELECT v.department,
ROUND(SUM(b.approved_amount) * 1.0 / SUM(b.billed_amount), 2) AS revenue_realization_ratio
FROM billing b
JOIN visits v ON b.visit_id = v.visit_id
GROUP BY v.department
ORDER BY revenue_realization_ratio ASC;
""", conn)

In [ ]:
# High Billed but Zero or Missing Approved Amount

pd.read_sql_query("""
SELECT b.bill_id,
b.visit_id,
b.billed_amount,
b.approved_amount
FROM billing b
WHERE b.billed_amount > 10000
AND (b.approved_amount IS NULL OR b.approved_amount = 0);
""", conn)

In [ ]:
# DATA QUALITY & INTEGRITY CHECKS
# Visits Without Billing Record

pd.read_sql_query("""
SELECT v.visit_id
FROM visits v
LEFT JOIN billing b ON v.visit_id = b.visit_id
WHERE b.visit_id IS NULL;
""", conn)

Indicates that there are zero visits without a billing record.

In [ ]:
# Billing Without Corresponding Visit

pd.read_sql_query("""
SELECT b.bill_id
FROM billing b
LEFT JOIN visits v ON b.visit_id = v.visit_id
WHERE v.visit_id IS NULL;
""", conn)

In [ ]:
# Duplicate Patient IDs

pd.read_sql_query("""
SELECT patient_id,
       COUNT(*) AS record_count
FROM patients
GROUP BY patient_id
HAVING COUNT(*) > 1;
""", conn)

In [ ]:
# Missing or Invalid Lenght of Stay or Payment Days

pd.read_sql_query("""
SELECT *
FROM visits
WHERE length_of_stay_hours IS NULL
   OR length_of_stay_hours < 0;
""", conn)

pd.read_sql_query("""
SELECT *
FROM billing
WHERE payment_days IS NULL
   OR payment_days < 0;
""", conn)

In [ ]:
# Visits linked to Patients Missing Insurance Provider

pd.read_sql_query("""
SELECT v.visit_id
FROM visits v
JOIN patients p ON v.patient_id = p.patient_id
WHERE p.insurance_provider IS NULL;
""", conn)